In [ ]:
from IPython.display import clear_output

var="/kaggle/input/vsdetection-packages-offline-installer-only/whls"
!pip install \
    "$var"/keras_nightly-3.12.0.dev2025100703-py3-none-any.whl \
      "$var"/tifffile-*.whl \
      "$var"/imagecodecs-*.whl \
    --no-index \
    --find-links "$var"

# Replace 'your-dataset-name' with the actual name of your uploaded Kaggle dataset
!pip install --no-index --find-links=/kaggle/input/datasets/sadek01/medicai-protobuf/wheels medicai protobuf
clear_output()


In [ ]:
import os, warnings

os.environ["KERAS_BACKEND"] = "torch"
os.environ["XLA_FLAGS"] = "--xla_gpu_force_compilation_parallelism=1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3" 
warnings.filterwarnings('ignore')

In [ ]:
import glob
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
root_dir = "/kaggle/input/vesuvius-npy"
images_dir = f"{root_dir}/train_images"
labels_dir = f"{root_dir}/train_labels"
test_images_dir="/kaggle/input/vesuvius-challenge-surface-detection/test_images"
all_image_files = sorted(glob.glob(os.path.join(images_dir, "*.npy")))
all_label_files = sorted(glob.glob(os.path.join(labels_dir, "*.npy")))
test_image_files = sorted(glob.glob(os.path.join(test_images_dir, "*.tif")))

# number of validation samples
val_count = 6

# Split by slicing
train_imgs = all_image_files[:-val_count]
val_imgs   = all_image_files[-val_count:]

train_lbls = all_label_files[:-val_count]
val_lbls   = all_label_files[-val_count:]

print("Train images:", len(train_imgs), len(train_lbls))
print("Val images:  ", len(val_imgs), len(val_lbls))
print("Test_images: ", len(test_image_files) )

In [ ]:
import keras
from keras import ops

import torch
import pytorch_lightning as pl

from pytorch_lightning.callbacks import ModelCheckpoint, LearningRateMonitor
from pytorch_lightning.callbacks import Callback

import medicai
from medicai.transforms import (
    Compose,
    NormalizeIntensity,
    ScaleIntensityRange,
    Resize,
    RandShiftIntensity,
    RandRotate90,
    RandRotate,
    RandFlip,
    RandCutOut,
    RandSpatialCrop
)
from medicai.layers import ResizingND
from medicai.models import (
    UNet, SegFormer, TransUNet, SwinUNETR, UPerNet, ConvNeXtV2Tiny, UNETRPlusPlus
)
from medicai.losses import (
    SparseDiceCELoss, SparseTverskyLoss, SparseCenterlineDiceLoss
)
from medicai.metrics import SparseDiceMetric
from medicai.callbacks import SlidingWindowInferenceCallback
from medicai.utils import SlidingWindowInference
from medicai.utils import soft_skeletonize

In [ ]:
keras.version(), keras.config.backend(), pl.__version__, medicai.version()

# Data Loader

In [ ]:
input_shape=(64, 128, 128)
batch_size=1
num_classes=3

# Total npy file - validation npy file
num_samples = 786 - val_count
epochs = 10

In [ ]:
# Assuming your dataset is named 'vesuvius-core-wheels'
!pip install --no-index --find-links=/kaggle/input/datasets/sadek01/wheelse/wheels batchgenerators monai

In [ ]:
import os
# Clone the repository


# Add the repository to your Python path so we can import from it
import sys
sys.path.append('/kaggle/input/datasets/sadek01/my-working-dir-backup/unetr_plus_plus')

import torch
import torch.nn as nn

# Add all necessary paths for the official repo imports to work
paths = [
    '/kaggle/input/datasets/sadek01/my-working-dir-backup/unetr_plus_plus',
    '/kaggle/input/datasets/sadek01/my-working-dir-backup/unetr_plus_plus/unetr_pp/network_architecture/synapse'
]
for p in paths:
    if p not in sys.path:
        sys.path.insert(0, p)

# Import the specific class
from unetr_pp_synapse import UNETR_PP

# Initialize with the 14-class settings from the Synapse checkpoint
model = UNETR_PP(
    in_channels=1,
    out_channels=14, 
    img_size=(64, 128, 128),
    feature_size=16,
    num_heads=4,
    depths=[3, 3, 3, 3],
    dims=[32, 64, 128, 256],
    do_ds=True,
)


in_ch = model.out1.conv.in_channels

model.out1.conv = nn.Conv3d(in_ch, 3, kernel_size=1, stride=1, bias=False)

# Ensure the new head is trainable
for param in model.out1.parameters():
    param.requires_grad = True

model.do_ds = False


In [ ]:
# loss function
dice_ce_loss_fn = SparseDiceCELoss(
    from_logits=False, 
    num_classes=num_classes,
    ignore_class_ids=2,
)
cldice_loss_fn = SparseCenterlineDiceLoss(
    from_logits=False, 
    num_classes=num_classes,
    target_class_ids=1,
    ignore_class_ids=2,
    iters=1, # ideal to set 20-50 - computationally expensive
)

def combined_loss(dice_ce_fn, cldice_fn):
    def loss_fn(y_true, y_pred):
        loss_dice_ce = dice_ce_fn(y_true, y_pred)
        loss_cldice = cldice_fn(y_true, y_pred)
        return loss_dice_ce + loss_cldice
    return loss_fn

In [ ]:
from monai.inferers import sliding_window_inference
import torch

In [ ]:
class VesuviusModel(pl.LightningModule):
    def __init__(
        self,
        model,
        learning_rate=1e-4,
        sw_batch_size=4,
        sw_overlap=0.5,
        num_classes=3,
        input_shape=(64, 128, 128)
    ):
        super().__init__()
        self.model = model
        self.num_classes = num_classes
        self.learning_rate = learning_rate
        self.sw_batch_size = sw_batch_size
        self.sw_overlap = sw_overlap
        
        # Loss functions (using the global instances you defined)
        self.train_criterion = combined_loss(
            dice_ce_loss_fn, cldice_loss_fn
        )
        self.val_criterion = dice_ce_loss_fn

        # Metric functions
        self.train_dice_score = SparseDiceMetric(
            from_logits=False,
            num_classes=num_classes,
            ignore_class_ids=2,
            name='dice',
        )
        self.val_dice_score = SparseDiceMetric(
            from_logits=False,
            num_classes=num_classes,
            ignore_class_ids=2,
            name='val_dice',
        )
        
        # Sliding window inferer
        self.sliding_window_inferer = SlidingWindowInference(
            self.model,
            num_classes=self.num_classes,
            roi_size=input_shape,
            sw_batch_size=self.sw_batch_size,
            overlap=self.sw_overlap,
        )
        self.save_hyperparameters(ignore=['model'])

    def forward(self, x):
        # Model returns PyTorch format: [B, 3, 64, 128, 128]
        return self.model(x)

    def training_step(self, batch, batch_idx):
        images, masks = batch 
        
        # 1. Forward Pass
        logits = self.forward(images)
        if isinstance(logits, (list, tuple)):
            logits = logits[0]
            
        # 2. BRIDGE: Convert to MedicAI format (NDHWC + Softmax)
        # Permute: [B, C, D, H, W] -> [B, D, H, W, C]
        prob = logits.permute(0, 2, 3, 4, 1)
        prob = torch.softmax(prob, dim=-1)
        
        # 3. BRIDGE: Convert Masks (B, 1, D, H, W) -> (B, D, H, W, 1)
        y_true = masks.permute(0, 2, 3, 4, 1).float()
        
        # 4. Calculate Loss & Metrics
        loss = self.train_criterion(y_true, prob)
        self.train_dice_score.update_state(y_true, prob)
 
        # Logging
        current_dice_score = self.train_dice_score.result()
        self.log('train_loss', loss, on_step=True, on_epoch=True, prog_bar=True)
        self.log('train_dice', current_dice_score, on_step=True, on_epoch=True, prog_bar=True)
        
        return loss

    def validation_step(self, batch, batch_idx):
        image, mask = batch # image: [B, 1, D, H, W]
        
        # 1. MONAI Sliding Window (Handles tiling on GPU)
        # We pass self.model directly; MONAI calls model(patch)
        output = sliding_window_inference(
            inputs=image, 
            roi_size=(64, 128, 128), # <-- Use your actual patch dimensions here
            sw_batch_size=4, 
            predictor=self.model,
            overlap=0.25,
            mode="gaussian"
        )
        
        # 2. Convert to Probabilities and permute to NDHWC
        prob = torch.softmax(output, dim=1).permute(0, 2, 3, 4, 1)
        y_true = mask.permute(0, 2, 3, 4, 1).float()
        
        # 3. Update Metrics and Loss
        loss = self.val_criterion(y_true, prob)
        self.val_dice_score.update_state(y_true, prob)
    
        # Logging
        self.log('val_loss', loss, on_epoch=True, prog_bar=True)
        self.log('val_dice', self.val_dice_score.result(), on_epoch=True, prog_bar=True)
        
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, self.parameters()), 
            lr=self.learning_rate,
            weight_decay=1e-5
        )

        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode='max',
            factor=0.5,
            patience=2,
        )
        return {
            'optimizer': optimizer,
            'lr_scheduler': {
                'scheduler': scheduler,
                'monitor': 'val_dice',
                'frequency': 1
            }
        }

    def on_train_epoch_end(self):
        self.train_dice_score.reset_state()

    def on_validation_epoch_end(self):
        self.val_dice_score.reset_state()
    def predict_step(self, batch, batch_idx, dataloader_idx=0):
        # 1. Get image
        image = batch[0] if isinstance(batch, (list, tuple)) else batch
        
        # 2. MONAI Sliding Window Inference
        output = sliding_window_inference(
            inputs=image, 
            roi_size=self.hparams.input_shape, 
            sw_batch_size=self.sw_batch_size, 
            predictor=self.model,
            overlap=self.sw_overlap,
            mode="gaussian",
            device="cpu" 
        )
        
        # 3. POST-PROCESSING: Filter for classes 0 and 1
        # output is [B, C, D, H, W] -> Slicing C to keep only indices 0 and 1
        # This effectively ignores class 2 entirely
        output_filtered = output[:, :2, ...] 
        
        # 4. Get the best class among the two
        prob = torch.softmax(output_filtered, dim=1)
        mask = torch.argmax(prob, dim=1) # Results in [B, D, H, W] with values {0, 1}
        
        # 5. Return as NumPy
        return mask.squeeze().detach().cpu().numpy().astype(np.uint8)   

**Custom Print Callback**

In [ ]:
class MetricLoggerCallback(Callback):
    def __init__(self, print_every_n_steps=20):
        super().__init__()
        self.print_every_n_steps = print_every_n_steps

    def on_train_batch_end(
        self, trainer, pl_module, outputs, batch, batch_idx
    ):
        global_step = trainer.global_step
        if global_step > 0 and (global_step % self.print_every_n_steps == 0):
            log_metrics = trainer.callback_metrics 
            loss = log_metrics.get('train_loss_step')
            dice = log_metrics.get('train_dice_step')
            print(
                f"[step {global_step}] batch log | "
                f"loss: {loss:.4f}, dice: {dice:.4f}"
            )

    def on_train_epoch_end(self, trainer, pl_module):
        log_metrics = trainer.callback_metrics 
        loss = log_metrics.get('train_loss_epoch')
        dice = log_metrics.get('train_dice_epoch')
        print(
            f"train loss: {loss:.4f}, train dice: {dice:.4f}"
        )
        print("-" * 50)

    def on_validation_epoch_end(self, trainer, pl_module):
        val_loss_key = 'val_loss' 
        val_dice_key = 'val_dice'
        val_loss = (
            trainer.callback_metrics.get(val_loss_key) or 
            trainer.callback_metrics.get(f'{val_loss_key}_epoch')
        )
        val_dice = (
            trainer.callback_metrics.get(val_dice_key) or 
            trainer.callback_metrics.get(f'{val_dice_key}_epoch')
        )
        print(
            f"epoch {trainer.current_epoch + 1:02d} completed: "
            f"val loss: {val_loss:.4f}, val dice: {val_dice:.4f}"
        )

In [ ]:
checkpoint_path = "/kaggle/input/datasets/sadek01/model-check/best-143-0.7600.ckpt"

lmodel = VesuviusModel.load_from_checkpoint(
    checkpoint_path,
    model=model,            # Pass the UNETR++ architecture
    learning_rate=1e-4,     # You can override parameters here
    sw_batch_size=2,
    sw_overlap=0.5
)


In [ ]:
def predict_with_tta(inputs, swi):
    logits = []

    # Original
    logits.append(swi(inputs))

    # Flips (spatial only)
    for axis in [1, 2, 3]:
        img_f = np.flip(inputs, axis=axis)
        p = swi(img_f)
        p = np.flip(p, axis=axis)
        logits.append(p)

    # Axial rotations (H, W)
    for k in [1, 2, 3]:
        img_r = np.rot90(inputs, k=k, axes=(2, 3))
        p = swi(img_r)
        p = np.rot90(p, k=-k, axes=(2, 3))
        logits.append(p)

    mean_logits = np.mean(logits, axis=0)
    mean_prob = ops.softmax(mean_logits, axis=-1)
    return mean_prob.argmax(-1).astype(np.uint8).squeeze()

In [ ]:

def build_anisotropic_struct(z_radius: int, xy_radius: int):
    z, r = z_radius, xy_radius
    if z == 0 and r == 0:
        return None
    if z == 0 and r > 0:
        size = 2 * r + 1
        struct = np.zeros((1, size, size), dtype=bool)
        cy, cx = r, r
        for dy in range(-r, r + 1):
            for dx in range(-r, r + 1):
                if dy * dy + dx * dx <= r * r:
                    struct[0, cy + dy, cx + dx] = True
        return struct
    if z > 0 and r == 0:
        struct = np.zeros((2 * z + 1, 1, 1), dtype=bool)
        struct[:, 0, 0] = True
        return struct
    depth = 2 * z + 1
    size = 2 * r + 1
    struct = np.zeros((depth, size, size), dtype=bool)
    cz, cy, cx = z, r, r
    for dz in range(-z, z + 1):
        for dy in range(-r, r + 1):
            for dx in range(-r, r + 1):
                if dy * dy + dx * dx <= r * r:
                    struct[cz + dz, cy + dy, cx + dx] = True
    return struct

def topo_postprocess(
    probs,
    T_low=0.90,
    T_high=0.90,
    z_radius=1,
    xy_radius=0,
    dust_min_size=100,
):
    # Step 1: 3D Hysteresis
    strong = probs >= T_high
    weak   = probs >= T_low

    if not strong.any():
        return np.zeros_like(probs, dtype=np.uint8)

    struct_hyst = ndi.generate_binary_structure(3, 3)
    mask = ndi.binary_propagation(
        strong, mask=weak, structure=struct_hyst
    )

    if not mask.any():
        return np.zeros_like(probs, dtype=np.uint8)

    # Step 2: 3D Anisotropic Closing
    if z_radius > 0 or xy_radius > 0:
        struct_close = build_anisotropic_struct(z_radius, xy_radius)
        if struct_close is not None:
            mask = ndi.binary_closing(mask, structure=struct_close)

    # Step 3: Dust Removal
    if dust_min_size > 0:
        mask = remove_small_objects(
            mask.astype(bool), min_size=dust_min_size
        )

    return mask.astype(np.uint8)

In [ ]:
def inference_pipelines(
    volume,
    T_low=0.50,
    T_high=0.90,
    z_radius=1,
    xy_radius=0,
    dust_min_size=100,
):
    probs = predict_with_tta(volume, swi)
    final = topo_postprocess(
        probs,
        T_low=T_low,
        T_high=T_high,
        z_radius=z_radius,
        xy_radius=xy_radius,
        dust_min_size=dust_min_size,
    )
    return final

In [ ]:
import pandas as pd
test_df = pd.read_csv(f"/kaggle/input/vesuvius-challenge-surface-detection/test.csv")
test_df.head()
test_dir="/kaggle/input/vesuvius-challenge-surface-detection/test_images"
zip_path="submission.zip"
test_images_dir="/kaggle/input/vesuvius-challenge-surface-detection/test_images"
test_image_files = sorted(glob.glob(os.path.join(test_images_dir, "*.tif")))

In [ ]:
def val_transformation(image):
    data = {"image": image}
    pipeline = Compose([
        NormalizeIntensity(
            keys=["image"], 
            nonzero=True,
            channel_wise=False
        ),
    ])
    result = pipeline(data)
    return result["image"]

In [ ]:
root_dir = "/kaggle/input/vesuvius-challenge-surface-detection"
test_dir = f"{root_dir}/test_images"
output_dir = "/kaggle/working/submission_masks"
zip_path = "/kaggle/working/submission.zip"
os.makedirs(output_dir, exist_ok=True)
test_df = pd.read_csv(f"{root_dir}/test.csv")
test_df.head()


In [ ]:
import torch
import numpy as np
import zipfile
import os
import tifffile
import gc
from scipy import ndimage as ndi
from skimage.morphology import remove_small_objects

# --- 1. Setup Device ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- 2. TTA Wrapper ---
def predict_with_tta(inputs, swi):
    """
    inputs: Tensor [1, 1, D, H, W] on 'device'
    swi: The sliding window function
    """
    total_probs = None
    count = 0

    # 1. Original
    p = swi(inputs) 
    total_probs = torch.softmax(p, dim=1)[:, 1, ...] 
    count += 1

    # 2. Spatial Flips
    for dims in [(2,), (3,), (4,)]:
        img_f = torch.flip(inputs, dims=dims)
        p_f = swi(img_f)
        p_f = torch.flip(p_f, dims=dims)
        total_probs += torch.softmax(p_f, dim=1)[:, 1, ...]
        count += 1

    # 3. Axial Rotations (H, W)
    for k in [1, 2, 3]:
        img_r = torch.rot90(inputs, k=k, dims=(3, 4))
        p_r = swi(img_r)
        p_r = torch.rot90(p_r, k=-k, dims=(3, 4))
        total_probs += torch.softmax(p_r, dim=1)[:, 1, ...]
        count += 1

    # Average Probs [D, H, W]
    avg_probs = (total_probs / count).squeeze().detach().cpu().numpy()
    return avg_probs

# --- 3. Prepare Model ---
lmodel.eval()
lmodel.to(device) 

def swi_fn(x):
    with torch.no_grad():
        res = sliding_window_inference(
            inputs=x.to(device), 
            roi_size=lmodel.hparams.input_shape, 
            sw_batch_size=lmodel.sw_batch_size, 
            predictor=lmodel.model, 
            overlap=lmodel.sw_overlap,
            mode="gaussian",
            device=device # Dynamic device assignment
        )
        return res[:, :2, ...] 

print(f"Starting inference on {len(test_image_files)} files...")

# --- 4. Main Submission Loop ---
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for image_id in test_df["id"]:
        tif_path = f"{test_dir}/{image_id}.tif"
        print(f"Processing: {image_id}")
        
        # 1. Load & Transform
        volume_np = tifffile.imread(tif_path).astype(np.float32) 
        volume_tf = val_transformation(volume_np)
        
        # Convert to NumPy then to PyTorch on correct device
        volume_np_transformed = volume_tf.numpy() 
        volume_tensor = torch.from_numpy(volume_np_transformed[None, None, ...]).to(device)
        
        # 2. Run Inference (TTA)
        probs = predict_with_tta(volume_tensor, swi_fn)

        # 3. Post-Process
        final_mask = topo_postprocess(
            probs,
            T_low=0.50, 
            T_high=0.90, 
            z_radius=1, 
            xy_radius=0, 
            dust_min_size=100
        )

        # 4. Save to Disk, Zip, and Delete
        out_path = f"{output_dir}/{image_id}.tif"
        tifffile.imwrite(out_path, final_mask.astype(np.uint8))

        z.write(out_path, arcname=f"{image_id}.tif")
        os.remove(out_path) 
        
        # 5. Thorough Memory Cleanup
        del volume_np, volume_tensor, probs, final_mask
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()

print("\nSubmission ZIP completed at:", zip_path)